<div style="background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 35px; border-radius: 12px; color: white; margin-bottom: 25px;">
  <h1 style="margin: 0 0 8px 0; font-size: 2em;">🏦 Lab 02: Create Your First Client Advisor Agent</h1>
  <p style="margin: 0; font-size: 1.15em; opacity: 0.92;">Zava Bank Workshop — Microsoft Foundry Agent Observability</p>
</div>

<div style="background-color: #e8f4fd; border-left: 4px solid #0078d4; padding: 1.2rem 1.5rem; border-radius: 6px; margin-bottom: 1.5rem;">

## What We Learn

Create a **Prompt Agent** using the Azure AI Projects SDK — the Zava Bank Client Advisor.

- Define an agent with financial advisory instructions and compliance guardrails
- Start conversations and send messages through the Responses API
- Handle **multi-turn interactions** about portfolios and market conditions
- Inspect the response object to understand what the model returns

By the end of this lab, you'll have a working conversational agent that can discuss financial topics in a professional, compliance-aware manner — the foundation we'll extend with real data tools in Lab 03.

</div>

<div style="background-color: #fff8e1; border-left: 4px solid #ffb300; padding: 1.2rem 1.5rem; border-radius: 6px; margin-bottom: 1.5rem;">

## 🏦 The Zava Bank Story

The advisory team needs a **concierge agent** — a knowledgeable front-desk assistant that understands client portfolios, risk profiles, and market conditions.

Think of the best financial advisor you've worked with: analytical, compliant, and able to discuss complex financial topics in plain language. That's what we're building. Not a system that replaces the advisor, but one that **augments** them — preparing briefings, flagging risk concerns, and synthesizing market context before every client meeting.

In this lab we give the agent its personality and compliance framework. It can *talk* about finance, but it doesn't have data access yet — that comes in Lab 03.

</div>

---
## Mental Model: Real-World Advisor → SDK Components

| Real-World Concept | SDK Component | What It Does |
|---|---|---|
| The advisor's expertise & personality | `instructions` (system prompt) | Defines how the agent thinks, what it prioritizes, and its compliance boundaries |
| The advisor sitting at their desk | `PromptAgentDefinition` + `create_version()` | Registers the agent with Foundry so it exists as a managed resource |
| A client walking in and starting a conversation | `openai_client.responses.create()` | Sends the first message and gets the agent's response |
| A back-and-forth discussion | Passing `previous_response_id` | Maintains conversational context across multiple turns |
| Reviewing the advisor's notes | `response.output`, `response.usage` | Inspect what the agent said, token consumption, and metadata |

---
## 1 · Environment Setup

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

# Walk up to find the .env file
notebook_dir = Path.cwd()
env_path = notebook_dir
while env_path != env_path.parent:
    if (env_path / ".env").exists():
        break
    env_path = env_path.parent

load_dotenv(env_path / ".env", override=True)

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

print(f"Project endpoint : {PROJECT_ENDPOINT[:60]}..." if len(PROJECT_ENDPOINT) > 60 else f"Project endpoint : {PROJECT_ENDPOINT}")
print(f"Model deployment : {MODEL_DEPLOYMENT}")

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
openai_client = project_client.get_openai_client()

print(f"✅ AIProjectClient connected")
print(f"✅ OpenAI client ready")

---
## 2 · Define the Advisor Instructions

The system prompt is the most important part of any agent. It defines the agent's identity, capabilities, communication style, and — critically for financial services — its compliance boundaries.

In [ ]:
ADVISOR_INSTRUCTIONS = """
You are the Zava Bank Client Advisor, an AI-powered assistant that supports
financial advisors at Zava Bank, a premium wealth-management firm.

## Role
You are an advisor-assist tool — you help human advisors prepare for client
meetings, review portfolios, and think through market conditions. You do NOT
interact directly with clients. Everything you produce is reviewed by a
licensed advisor before reaching a client.

## Capabilities
Zava Bank can help advisors with:
- Portfolio analysis — holdings, allocation, performance, unrealized P&L
- Risk assessment — beta, Sharpe ratio, concentration, drawdown exposure
- Market research — indices, sector trends, economic indicators

## Communication style
- Be analytical and data-driven — lead with numbers when you have them
- Discuss complex financial topics in plain, professional language
- Structure longer responses with clear sections and bullet points
- When asked about a client, frame your response as a briefing for the advisor

## Compliance rules (non-negotiable)
- Never recommend specific real stocks, funds, or securities by name
- Never execute, simulate, or suggest executing trades
- Never guarantee future returns or make promissory statements
- Always include appropriate disclaimers when discussing investment topics
- If you don't have data to answer a question, say so — do not fabricate
- Recommend professional consultation for tax, legal, or estate matters

## Disclaimer
End substantive advisory responses with:
"Note: This analysis is generated by an AI assistant and should be reviewed
by a licensed advisor before any client communication or action."
""".strip()

print(f"Instructions defined — {len(ADVISOR_INSTRUCTIONS)} characters")
print(f"First 200 chars: {ADVISOR_INSTRUCTIONS[:200]}...")

<div style="background: #f5f5f5; border-left: 4px solid #533483; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**Why this prompt matters:** In financial services, the system prompt is your first line of defense. It defines what the agent *will* and *won't* do. The compliance rules above aren't suggestions — they're hard boundaries. In Lab 06 (Red Teaming), we'll test whether these boundaries actually hold up under adversarial pressure.

</div>

---
## 3 · Create the Agent

We register the agent with Foundry using `create_version`. This creates a managed agent resource — Foundry stores the definition, and we get back a versioned reference we can use for conversations.

In [ ]:
agent = project_client.agents.create_version(
    agent_name="zava-bank-client-advisor",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=ADVISOR_INSTRUCTIONS,
    ),
)

print(f"✅ Agent created")
print(f"   Name    : {agent.name}")
print(f"   Version : {agent.version}")
print(f"   Model   : {MODEL_DEPLOYMENT}")

<div style="background: #f5f5f5; border-left: 4px solid #0f3460; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**What just happened?** We sent a `PromptAgentDefinition` to Foundry. The platform now holds:
- The system instructions (our compliance-aware prompt)
- The model deployment to use (gpt-4o or gpt-4.1)
- A versioned agent name we can reference in conversations

No tools yet — this agent can only reason from its training data and our instructions. We add data tools in Lab 03.

</div>

---
## 4 · Start a Conversation

Time to talk to the advisor. We send a message through the Responses API and see what comes back.

In [ ]:
first_message = (
    "Hi! I need to review my client Margaret Chen's account before our "
    "quarterly meeting. What should I be looking at?"
)

print(f"👤 Advisor asks: {first_message}\n")

response_1 = openai_client.responses.create(
    model=MODEL_DEPLOYMENT,
    instructions=ADVISOR_INSTRUCTIONS,
    input=first_message,
)

# Extract the text output
advisor_reply_1 = "\n".join(
    item.text for item in response_1.output if hasattr(item, "text")
)

print(f"🏦 Zava Advisor:\n{advisor_reply_1}")

<div style="background: #f5f5f5; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**Notice:** The agent doesn't have access to real client data yet — it can only describe what an advisor *should* look at in general terms. This is intentional. A well-designed agent should be useful even without tools, providing a structured framework for thinking. In Lab 03 we connect it to actual portfolio, risk, and market data.

</div>

---
## 5 · Multi-Turn Conversation

Real advisory conversations are multi-turn. The advisor asks a question, reviews the response, then digs deeper. We maintain context by passing `previous_response_id` so the model sees the full conversation history.

In [ ]:
# Turn 2: Ask about risk factors
second_message = "What are the key risk factors I should discuss with her?"

print(f"👤 Advisor asks: {second_message}\n")

response_2 = openai_client.responses.create(
    model=MODEL_DEPLOYMENT,
    instructions=ADVISOR_INSTRUCTIONS,
    input=second_message,
    previous_response_id=response_1.id,
)

advisor_reply_2 = "\n".join(
    item.text for item in response_2.output if hasattr(item, "text")
)

print(f"🏦 Zava Advisor:\n{advisor_reply_2}")

In [ ]:
# Turn 3: Market context for a conservative portfolio
third_message = (
    "How should I position current market conditions in the context of "
    "her conservative portfolio?"
)

print(f"👤 Advisor asks: {third_message}\n")

response_3 = openai_client.responses.create(
    model=MODEL_DEPLOYMENT,
    instructions=ADVISOR_INSTRUCTIONS,
    input=third_message,
    previous_response_id=response_2.id,
)

advisor_reply_3 = "\n".join(
    item.text for item in response_3.output if hasattr(item, "text")
)

print(f"🏦 Zava Advisor:\n{advisor_reply_3}")

<div style="background: #f5f5f5; border-left: 4px solid #533483; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**Multi-turn in action:** Each response builds on the previous one. By Turn 3, the agent references Margaret Chen's conservative profile even though we only mentioned it in Turn 1. The `previous_response_id` parameter tells the Responses API to chain the conversation — the platform manages the thread history for you.

</div>

---
## 6 · Explore the Response Object

Before we move on, let's look inside the response. Understanding what the API returns helps with debugging, logging, and building production integrations.

In [ ]:
print("=" * 60)
print("Response Object Anatomy")
print("=" * 60)

print(f"\nresponse.id       : {response_3.id}")
print(f"response.model    : {response_3.model}")
print(f"response.status   : {response_3.status}")

print(f"\n--- Token Usage ---")
if response_3.usage:
    print(f"  Input tokens    : {response_3.usage.input_tokens:,}")
    print(f"  Output tokens   : {response_3.usage.output_tokens:,}")
    print(f"  Total tokens    : {response_3.usage.total_tokens:,}")

print(f"\n--- Output Items ---")
for i, item in enumerate(response_3.output):
    print(f"  [{i}] type={item.type}")
    if hasattr(item, "text"):
        print(f"      text preview: {item.text[:100]}...")
    if hasattr(item, "name"):
        print(f"      tool name: {item.name}")

<div style="background: #f5f5f5; border-left: 4px solid #0f3460; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**Key fields to know:**

| Field | Why It Matters |
|---|---|
| `response.id` | Unique identifier — use this for `previous_response_id` to chain turns |
| `response.model` | Confirms which model deployment handled the request |
| `response.status` | Should be `completed` — watch for `failed` or `incomplete` in production |
| `response.usage` | Token counts — critical for cost tracking and rate-limit planning |
| `response.output` | List of output items — text responses, function calls, or both |

In Lab 03, `response.output` will include `function_call` items when the agent decides to use tools. For now, it only contains text.

</div>

---
## 7 · Cleanup

Delete the agent version and close clients. In a production setting you'd keep the agent; here we clean up to avoid clutter in your Foundry project.

In [ ]:
# Delete the agent version
project_client.agents.delete_version(
    agent_name="zava-bank-client-advisor",
    version=agent.version,
)
print(f"✅ Deleted agent version: zava-bank-client-advisor v{agent.version}")

# Close clients
project_client.close()
print("✅ Clients closed")

---

<div style="background: linear-gradient(135deg, #0f3460, #533483); padding: 20px 25px; border-radius: 10px; color: white; margin-top: 25px;">

## ✅ Lab 02 Summary

We built the **Zava Bank Client Advisor** — a prompt agent with a compliance-aware personality, professional communication style, and clear operational boundaries.

**What we covered:**
- **System prompt design** — instructions that define identity, capabilities, and compliance guardrails
- **Agent registration** — `create_version()` with `PromptAgentDefinition` to create a managed Foundry resource
- **Single-turn interaction** — sending a message and getting a structured response
- **Multi-turn conversation** — chaining turns with `previous_response_id` for contextual follow-ups
- **Response inspection** — understanding the anatomy of what the API returns

**What the agent can't do yet:** access real data. Ask it about Margaret Chen's portfolio and it gives general advice. In **Lab 03**, we add `FunctionTool` objects backed by Zava Bank's portfolio, risk, and market data — turning general conversation into grounded, data-driven advisory.

</div>